# RideWise — Data Preprocessing

Fix all data-quality issues and save clean datasets for feature engineering.

## Setup

In [10]:
import os, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
%matplotlib inline
sns.set_theme(style="whitegrid", font_scale=1.2)

DATA_RAW       = ".."                    # INTERNSHIP folder (one level up)
DATA_PROCESSED = os.path.join("..", "processed")   # saves to INTERNSHIP/processed/
os.makedirs(DATA_PROCESSED, exist_ok=True)
print("Directories ready ✓")
print("Raw data from   :", os.path.abspath(DATA_RAW))
print("Processed to    :", os.path.abspath(DATA_PROCESSED))


Directories ready ✓
Raw data from   : c:\Users\Admin\Desktop\DATA SCIENCE\INTERNSHIP
Processed to    : c:\Users\Admin\Desktop\DATA SCIENCE\INTERNSHIP\processed


## Load Raw Data

In [11]:
riders     = pd.read_csv(os.path.join(DATA_RAW, "riders.csv"),     parse_dates=["signup_date"])
trips      = pd.read_csv(os.path.join(DATA_RAW, "trips.csv"))
drivers    = pd.read_csv(os.path.join(DATA_RAW, "drivers.csv"),    parse_dates=["signup_date"])
sessions   = pd.read_csv(os.path.join(DATA_RAW, "sessions.csv"))
promotions = pd.read_csv(os.path.join(DATA_RAW, "promotions.csv"))

for name, df in [("riders", riders), ("trips", trips), ("drivers", drivers),
                 ("sessions", sessions), ("promotions", promotions)]:
    print(f"  {name:<12} {df.shape[0]:>7,} rows  {df.shape[1]:>2} cols")


  riders        10,000 rows   8 cols
  trips        200,000 rows  16 cols
  drivers        5,000 rows   7 cols
  sessions      50,000 rows   8 cols
  promotions        20 rows  11 cols


## Step 1 · Clean Riders

In [12]:
riders_clean = riders.copy()

# Round age to whole number
riders_clean["age"] = riders_clean["age"].round().astype("Int64")

# Convert churn probability to binary label (threshold = 0.5)
riders_clean["churned"]     = (riders_clean["churn_prob"] > 0.5).astype(int)
riders_clean = riders_clean.drop(columns=["churn_prob"])

# Encode referral as binary signal
riders_clean["was_referred"] = riders_clean["referred_by"].notna().astype(int)
riders_clean = riders_clean.drop(columns=["referred_by"])

print("Riders clean shape:", riders_clean.shape)
riders_clean.head()


Riders clean shape: (10000, 8)


,user_id,signup_date,loyalty_status,age,city,avg_rating_given,churned,was_referred
0,R00000,2025-01-24,Bronze,35,Nairobi,5.0,0,1
1,R00001,2024-09-09,Bronze,35,Nairobi,4.7,1,0
2,R00002,2024-09-07,Bronze,47,Lagos,4.2,1,0
3,R00003,2025-03-17,Bronze,42,Nairobi,4.9,0,0
4,R00004,2024-08-20,Silver,41,Lagos,3.9,0,1


## Step 2 · Clean Trips

In [13]:
trips_clean = trips.copy()

# Parse timezone-aware timestamps and strip timezone
for col in ["pickup_time", "dropoff_time"]:
    trips_clean[col] = pd.to_datetime(trips_clean[col], utc=True).dt.tz_convert(None)

trips_clean = trips_clean.dropna(subset=["pickup_time", "dropoff_time"])

# Engineer features
trips_clean["trip_duration"] = (
    (trips_clean["dropoff_time"] - trips_clean["pickup_time"]).dt.total_seconds() / 60
)
trips_clean["total_revenue"] = (
    trips_clean["fare"] * trips_clean["surge_multiplier"] + trips_clean["tip"].fillna(0)
)
trips_clean["hour_of_day"]   = trips_clean["pickup_time"].dt.hour
trips_clean["day_of_week"]   = trips_clean["pickup_time"].dt.day_name()

# Remove bad rows
trips_clean = trips_clean[trips_clean["trip_duration"] > 0]
trips_clean["tip"] = trips_clean["tip"].fillna(0)

print("Trips clean shape:", trips_clean.shape)
trips_clean.head()


Trips clean shape: (200000, 20)


,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,pickup_lng,dropoff_lat,dropoff_lng,weather,city,loyalty_status,trip_duration,total_revenue,hour_of_day,day_of_week
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 16:14:50,2024-11-27 17:06:50,-1.108123,36.912209,-1.068155,36.875377,Foggy,Nairobi,Bronze,52.0,12.11,16,Wednesday
1,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 22:59:48,2024-10-28 23:12:48,6.675266,3.515740,6.641734,3.525620,Sunny,Lagos,Gold,13.0,8.75,22,Monday
2,T000002,R00567,D02035,19.68,1.0,0.00,Card,2025-02-17 03:09:41,2025-02-17 03:25:41,-1.248589,37.010668,-1.273182,37.018586,Cloudy,Nairobi,Bronze,16.0,19.68,3,Monday
3,T000003,R09573,D02657,16.43,1.0,0.01,Mobile Money,2024-06-18 17:22:14,2024-06-18 17:27:14,29.819554,31.188780,29.837689,31.232978,Cloudy,Cairo,Bronze,5.0,16.44,17,Tuesday
4,T000004,R03446,D01026,8.70,1.0,1.06,Card,2024-10-05 07:31:16,2024-10-05 08:01:16,-1.676479,36.729219,-1.638395,36.694063,Sunny,Nairobi,Gold,30.0,9.76,7,Saturday


## Step 3 · Clean Drivers

In [14]:
drivers_clean = drivers.copy()

drivers_clean["rating"]          = drivers_clean["rating"].fillna(drivers_clean["rating"].median())
drivers_clean["acceptance_rate"] = (
    drivers_clean["acceptance_rate"].fillna(drivers_clean["acceptance_rate"].median())
)

print("Drivers clean shape:", drivers_clean.shape)
print("Remaining nulls:\n", drivers_clean.isnull().sum())


Drivers clean shape: (5000, 7)
Remaining nulls:
 driver_id          0
rating             0
vehicle_type       0
signup_date        0
last_active        0
city               0
acceptance_rate    0
dtype: int64


## Step 4 · Clean Sessions

In [15]:
sessions_clean = sessions.copy()
sessions_clean["session_time"] = (
    pd.to_datetime(sessions_clean["session_time"], utc=True).dt.tz_convert(None)
)
sessions_clean = sessions_clean.dropna(subset=["session_time"])

print("Sessions clean shape:", sessions_clean.shape)
sessions_clean.head()


Sessions clean shape: (50000, 8)


,session_id,rider_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 16:52:06,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 05:05:22,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 21:12:25,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:26:25,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:17:22,4,1,0,Lagos,Bronze


## Step 5 · Referential Integrity Check

In [16]:
valid_riders  = set(riders_clean["user_id"])
valid_drivers = set(drivers_clean["driver_id"])

before_trips    = len(trips_clean)
before_sessions = len(sessions_clean)

trips_clean    = trips_clean[trips_clean["user_id"].isin(valid_riders)]
trips_clean    = trips_clean[trips_clean["driver_id"].isin(valid_drivers)]
sessions_clean = sessions_clean[sessions_clean["rider_id"].isin(valid_riders)]

print(f"Trips dropped    : {before_trips    - len(trips_clean):,}")
print(f"Sessions dropped : {before_sessions - len(sessions_clean):,}")
print("Referential integrity ✓")


Trips dropped    : 0
Sessions dropped : 0
Referential integrity ✓


## Step 6 · Validate & Save

In [17]:
print(f"{'Dataset':<12}  {'Raw':>8}  {'Clean':>8}  {'Dropped':>8}")
print("-" * 42)
for name, raw, clean in [
    ("riders",   riders,   riders_clean),
    ("drivers",  drivers,  drivers_clean),
    ("trips",    trips,    trips_clean),
    ("sessions", sessions, sessions_clean),
]:
    print(f"{name:<12}  {len(raw):>8,}  {len(clean):>8,}  {len(raw)-len(clean):>8,}")


Dataset            Raw     Clean   Dropped
------------------------------------------
riders          10,000    10,000         0
drivers          5,000     5,000         0
trips          200,000   200,000         0
sessions        50,000    50,000         0


In [18]:
files = {
    "riders_clean.csv":   riders_clean,
    "trips_clean.csv":    trips_clean,
    "drivers_clean.csv":  drivers_clean,
    "sessions_clean.csv": sessions_clean,
}

for fname, df in files.items():
    df.to_csv(os.path.join(DATA_PROCESSED, fname), index=False)
    print(f"Saved  {fname}  ({len(df):,} rows)")


Saved  riders_clean.csv  (10,000 rows)
Saved  trips_clean.csv  (200,000 rows)
Saved  drivers_clean.csv  (5,000 rows)
Saved  sessions_clean.csv  (50,000 rows)
